
## Introduction

This notebook evaluates the performance of four machine learning regression models developed to predict menstrual cycle length.

The models evaluated are:

1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor
4. Support Vector Regression (SVR)

The models are evaluated using the following performance metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between actual and predicted cycle lengths.
- **Mean Squared Error (MSE):** Measures the average squared prediction error.
- **Root Mean Squared Error (RMSE):** Measures prediction error while giving greater importance to larger errors.
- **R² Score:** Measures how well the model explains the variation in menstrual cycle length.

Additionally, actual and predicted cycle lengths are compared, and a baseline regression model is used to determine whether machine learning models perform better than simply predicting the average cycle length.

The objective is to identify the model with the most reliable predictive performance.

In [ ]:
#import libraries 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR


In [ ]:

#Load Dataset 

df = pd.read_csv("../Dataset/menstrual_health_cleaned.csv")

X = df.drop(columns=[
    "Length_of_cycle",
    "Mean_of_length_of_cycle",
    "Estimated_day_of_ovulution",
    "Weight"
])

y = df["Length_of_cycle"]

categorical_features = ["Unusual_Bleeding"]

numerical_features = [
    "number_of_peak",
    "Age",
    "Length_of_Leutal_Phase",
    "Length_of_menses",
    "BMI",
    "Menses_score"
]



#Spliot the dataset 



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training records:", X_train.shape)
print("Testing records:", X_test.shape)

#Perform preprocessing

# One-hot encoding
encoder = OneHotEncoder(
    drop="if_binary",
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

# Standard scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train[numerical_features]
)

X_test_scaled = scaler.transform(
    X_test[numerical_features]
)


# Combine processed features
X_train_processed = np.concatenate(
    [X_train_scaled, X_train_encoded],
    axis=1
)

X_test_processed = np.concatenate(
    [X_test_scaled, X_test_encoded],
    axis=1
)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)


#define regression model

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ),

    "Support Vector Regression": SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1
    )
}

trained_models = {}

for name, model in models.items():

    model.fit(X_train_processed, y_train)

    trained_models[name] = model

    print(name, "training completed")




Training records: (64, 7)
Testing records: (17, 7)
Training data: (64, 7)
Testing data: (17, 7)
Linear Regression training completed
Random Forest training completed
Gradient Boosting training completed
Support Vector Regression training completed


In [ ]:
#Import evaluation metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd


#calculate evaluation metrics 

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test_processed)

    mae = mean_absolute_error(y_test, y_pred)

    mse = mean_squared_error(y_test, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)

print(results_df)


#identify the best performimng model 

best_model = results_df.loc[
    results_df["MAE"].idxmin()
]

print("Best model based on MAE:")
print(best_model)

                       Model       MAE        MSE      RMSE  R2 Score
0          Linear Regression  2.523500  11.821718  3.438273 -0.244165
1              Random Forest  3.543314  17.830861  4.222660 -0.876591
2          Gradient Boosting  3.580060  19.166316  4.377935 -1.017140
3  Support Vector Regression  2.321734  10.337237  3.215157 -0.087932
Best model based on MAE:
Model       Support Vector Regression
MAE                          2.321734
MSE                         10.337237
RMSE                         3.215157
R2 Score                    -0.087932
Name: 3, dtype: object


In [5]:
# Generate predictions using the trained models

predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test_processed)

print("Predictions generated successfully!")

#Compare actual and predicted values

comparison = pd.DataFrame({
    "Actual Cycle Length": y_test.values,
    "Predicted Cycle Length": predictions["Support Vector Regression"]
})

comparison["Absolute Error"] = abs(
    comparison["Actual Cycle Length"] -
    comparison["Predicted Cycle Length"]
)

print(comparison)

Predictions generated successfully!
    Actual Cycle Length  Predicted Cycle Length  Absolute Error
0                    30               28.894493        1.105507
1                    27               28.486579        1.486579
2                    35               28.097089        6.902911
3                    25               28.548112        3.548112
4                    28               29.054975        1.054975
5                    29               29.863737        0.863737
6                    30               28.821333        1.178667
7                    30               28.227324        1.772676
8                    35               28.845646        6.154354
9                    37               29.668922        7.331078
10                   28               29.510016        1.510016
11                   30               30.275970        0.275970
12                   30               28.726998        1.273002
13                   27               29.899119        2.899119
14  

In [3]:
#Compare against a baseline model

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train_processed, y_train)

baseline_predictions = baseline.predict(X_test_processed)

print("Baseline MAE:",
      mean_absolute_error(y_test, baseline_predictions))

print("Baseline R2:",
      r2_score(y_test, baseline_predictions))

Baseline MAE: 2.245404411764706
Baseline R2: -0.03001178205230337


### Conclusion

Four regression models were evaluated for menstrual cycle length prediction.

Among the four models, Support Vector Regression (SVR) achieved the lowest MAE of approximately 2.32 days and RMSE of 3.22 days.

However, all four models produced negative R² scores, indicating limited predictive performance on the test dataset.

A baseline model was also evaluated. It achieved an MAE of approximately 2.25 days, performing slightly better than SVR. This indicates that the current machine learning models have not demonstrated an improvement over the baseline approach.

The results may be influenced by the small dataset size (81 records), limited predictive features, and the variability of menstrual cycle patterns.

Further evaluation using 5-fold cross-validation is required to assess model stability and generalisation.

Overall, this evaluation highlights the importance of comparing machine learning models against simple baseline approaches rather than selecting a model based only on its relative performance among competing algorithms.